# Setup and logging
This notebook does two jobs:
1. Makes the schema (database) and all tables, if they don't exist yet.
2. Gives small helper functions that write one row to the log table every time a step runs.

Run this notebook by itself once. Other notebooks call it with `%run`.

In [0]:
dbutils.widgets.text("catalog", "workspace", "Catalog name")
dbutils.widgets.text("db_schema", "dav_project", "Schema (database) name")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("db_schema")

# Check your Catalog Explorer (left sidebar, Catalog icon) for the real catalog name
# on your workspace if "workspace" does not exist. Change the widget above, not this line.

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")

print(f"Using {CATALOG}.{SCHEMA}")

Using workspace.dav_project


## Bronze tables
One table per entity (commits, issues, pulls), plus one quarantine table for
records that could not even be given a key. Every table has `load_timestamp`.

In [0]:
BRONZE_ENTITY_DDL = """
CREATE TABLE IF NOT EXISTS {catalog}.{schema}.bronze_{entity} (
  record_key      STRING    NOT NULL,
  source_repo     STRING    NOT NULL,
  entity_type     STRING    NOT NULL,
  raw_json        STRING    NOT NULL,
  api_params      STRING,
  load_type       STRING    NOT NULL,
  batch_id        STRING    NOT NULL,
  run_id          STRING    NOT NULL,
  load_timestamp  TIMESTAMP NOT NULL
) USING DELTA
"""

for entity in ["commits", "issues", "pulls"]:
    spark.sql(BRONZE_ENTITY_DDL.format(catalog=CATALOG, schema=SCHEMA, entity=entity))
    print(f"ok: bronze_{entity}")

ok: bronze_commits
ok: bronze_issues
ok: bronze_pulls


In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.bronze_quarantine (
  source_repo     STRING,
  entity_type     STRING,
  raw_record      STRING,
  reason          STRING,
  batch_id        STRING,
  run_id          STRING,
  load_timestamp  TIMESTAMP
) USING DELTA
""")
print("ok: bronze_quarantine")

ok: bronze_quarantine


## Silver tables
Cleaned, typed, deduplicated. Author names and emails are hashed, not stored.
Pull requests are excluded from `silver_issue` (GitHub's issues endpoint
returns both; we split them using the `pull_request` field).

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.silver_commit (
  sha                 STRING    NOT NULL,
  repo                STRING    NOT NULL,
  author_key          STRING,
  committer_key       STRING,
  committed_at         TIMESTAMP,
  message_length       INT,
  is_merge_commit       BOOLEAN,
  load_timestamp        TIMESTAMP NOT NULL
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.silver_issue (
  issue_id             STRING    NOT NULL,
  repo                 STRING    NOT NULL,
  number                INT,
  state                 STRING,
  author_key            STRING,
  created_at            TIMESTAMP,
  updated_at            TIMESTAMP,
  closed_at             TIMESTAMP,
  comment_count          INT,
  time_to_close_hours    DOUBLE,
  load_timestamp         TIMESTAMP NOT NULL
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.silver_pull_request (
  pr_id                STRING    NOT NULL,
  repo                 STRING    NOT NULL,
  number                INT,
  state                 STRING,
  author_key            STRING,
  created_at            TIMESTAMP,
  updated_at            TIMESTAMP,
  closed_at             TIMESTAMP,
  merged_at             TIMESTAMP,
  is_merged              BOOLEAN,
  is_draft                BOOLEAN,
  base_branch            STRING,
  time_to_merge_hours     DOUBLE,
  load_timestamp          TIMESTAMP NOT NULL
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.silver_quarantine (
  entity_type      STRING,
  record_key        STRING,
  repo               STRING,
  reason              STRING,
  raw_json            STRING,
  batch_id             STRING,
  run_id               STRING,
  load_timestamp        TIMESTAMP
) USING DELTA
""")

print("ok: silver_commit, silver_issue, silver_pull_request, silver_quarantine")

ok: silver_commit, silver_issue, silver_pull_request, silver_quarantine


## Log table
One row per run of one step. This is the proof table your grader will look at.

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.pipeline_execution_logs (
  run_id             STRING    NOT NULL,
  layer               STRING    NOT NULL,
  entity                STRING,
  parameters              STRING,
  start_time               TIMESTAMP NOT NULL,
  end_time                 TIMESTAMP,
  status                     STRING,
  rows_inserted               BIGINT,
  rows_updated                 BIGINT,
  rows_quarantined               BIGINT,
  error_message                    STRING,
  rows_aggregated                    BIGINT,
  duration_seconds                     DOUBLE
) USING DELTA
""")
print("ok: pipeline_execution_logs")

ok: pipeline_execution_logs


## Phase 3 upgrade of the log table
The log table already exists from Phase 2, so `CREATE TABLE IF NOT EXISTS`
above does not change it. This cell adds the two new columns to an old
table, only if they are missing. Safe to run any number of times.
- `rows_aggregated`: how many source rows were read and rolled up (Silver to Gold).
- `duration_seconds`: how long the step took.

In [0]:
_log_table = f"{CATALOG}.{SCHEMA}.pipeline_execution_logs"
_have = [r["col_name"] for r in spark.sql(f"DESCRIBE TABLE {_log_table}").collect()]
for _col, _type in [("rows_aggregated", "BIGINT"), ("duration_seconds", "DOUBLE")]:
    if _col not in _have:
        spark.sql(f"ALTER TABLE {_log_table} ADD COLUMNS ({_col} {_type})")
        print(f"added column: {_col}")
    else:
        print(f"column already there: {_col}")

column already there: rows_aggregated
column already there: duration_seconds


## Small helper functions
`new_run_id()` makes one ID for one notebook run.
`log_run(...)` writes one row to the log table. Call it even when a step fails.
`get_merge_metrics(...)` reads how many rows a MERGE just inserted and updated.

In [0]:
import uuid
import json
from datetime import datetime, timezone
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType, DoubleType


def new_run_id() -> str:
    return str(uuid.uuid4())


def now_utc():
    return datetime.now(timezone.utc)


# Explicit schema for the log table writer. A plain list of column names
# makes Spark guess each column's type from the one row we give it, and it
# cannot guess the type of a column whose only value is None (this is
# exactly what caused the CANNOT_DETERMINE_TYPE error). A real StructType
# removes the guessing entirely, same as everywhere else in this pipeline.
LOG_SCHEMA = StructType([
    StructField("run_id", StringType(), False),
    StructField("layer", StringType(), False),
    StructField("entity", StringType(), True),
    StructField("parameters", StringType(), True),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("rows_inserted", LongType(), True),
    StructField("rows_updated", LongType(), True),
    StructField("rows_quarantined", LongType(), True),
    StructField("error_message", StringType(), True),
    StructField("rows_aggregated", LongType(), True),
    StructField("duration_seconds", DoubleType(), True),
])


def log_run(run_id, layer, entity, parameters: dict, start_time, end_time,
            status, rows_inserted=0, rows_updated=0, rows_quarantined=0,
            error_message=None, rows_aggregated=0):
    """Writes exactly one row to pipeline_execution_logs.

    duration_seconds is worked out here from start_time and end_time.
    If the log write itself fails, it prints a loud warning and carries on, so
    a logging problem never hides the result of the real work.
    """
    duration = None
    if start_time is not None and end_time is not None:
        duration = float((end_time - start_time).total_seconds())
    row = [(
        run_id, layer, entity, json.dumps(parameters),
        start_time, end_time, status,
        int(rows_inserted), int(rows_updated), int(rows_quarantined),
        error_message, int(rows_aggregated), duration,
    )]
    try:
        df = spark.createDataFrame(row, schema=LOG_SCHEMA)
        df.write.format("delta").mode("append").saveAsTable(
            f"{CATALOG}.{SCHEMA}.pipeline_execution_logs"
        )
        print(f"[log] {layer} / {entity} -> {status} "
              f"(inserted={rows_inserted}, updated={rows_updated}, "
              f"quarantined={rows_quarantined}, aggregated={rows_aggregated}, "
              f"seconds={duration if duration is None else round(duration, 1)})")
    except Exception as e:
        print(f"[log] WARNING: could not write the log row for {layer} / {entity}: {e}")


def get_merge_metrics(full_table_name: str) -> dict:
    """Reads the row counts from the most recent MERGE on a Delta table."""
    hist = spark.sql(f"DESCRIBE HISTORY {full_table_name} LIMIT 1").collect()[0]
    metrics = hist["operationMetrics"] or {}
    return {
        "inserted": int(metrics.get("numTargetRowsInserted", 0)),
        "updated": int(metrics.get("numTargetRowsUpdated", 0)),
    }


print("helpers ready: new_run_id, now_utc, log_run, get_merge_metrics")

helpers ready: new_run_id, now_utc, log_run, get_merge_metrics
